## Xenium QC Report Module

### Please enter the following information before running the notebook
1. uuid_list: If you are starting from a list of UUIDs, please add a comma separated list of UUIDs, where each UUID is within quotes.
2. project_short_name: Name of the project store where the aggregated file can be uploaded
3. title description for aggregated file output name
4. pdf_file_name: Filename for the pdf report of all visualizations
5. all_qc_report_path

In [1]:
from google.cloud import storage
from pathlib import Path

BASE_DIR = "/home/workspace"
BUCKET_NAME = "temp-xenium-hise-transfer"  # no gs:// prefix for the SDK

OUTPUT = Path(BASE_DIR) / "xenium/qc/EXP-02106-02107/csvs"
SAMPLES = "TSS12235-002, TSS12231-002, TSS10546-030, TSS12235-001, TSS12236-001, TSS12231-001, TSS12232-001, TSS10546-031"

sample_list = [s.strip() for s in SAMPLES.split(",")]
OUTPUT.mkdir(parents=True, exist_ok=True)

client = storage.Client()
bucket = client.bucket(BUCKET_NAME)

for sample_id in sample_list:
    print(f"Looking for {sample_id}...")

    matches = [
        blob for blob in bucket.list_blobs()
        if sample_id in blob.name and blob.name.endswith("metrics_summary.csv")
    ]

    if not matches:
        print(f"  ERROR: No match found for {sample_id}")
        continue

    if len(matches) > 1:
        print(f"  WARNING: Multiple matches for {sample_id}, using first: {matches[0].name}")

    dst = OUTPUT / f"{sample_id}_metric_summary.csv"
    matches[0].download_to_filename(str(dst))
    print(f"  Done -> {dst}")

/home/workspace/environment/minimal/lib/python3.11/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/workspace/environment/minimal/lib/python3.11/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Looking for TSS12235-002...
  Done -> /home/workspace/xenium/qc/EXP-02106-02107/csvs/TSS12235-002_metric_summary.csv
Looking for TSS12231-002...
  Done -> /home/workspace/xenium/qc/EXP-02106-02107/csvs/TSS12231-002_metric_summary.csv
Looking for TSS10546-030...
  Done -> /home/workspace/xenium/qc/EXP-02106-02107/csvs/TSS10546-030_metric_summary.csv
Looking for TSS12235-001...
  Done -> /home/workspace/xenium/qc/EXP-02106-02107/csvs/TSS12235-001_metric_summary.csv
Looking for TSS12236-001...
  Done -> /home/workspace/xenium/qc/EXP-02106-02107/csvs/TSS12236-001_metric_summary.csv
Looking for TSS12231-001...
  Done -> /home/workspace/xenium/qc/EXP-02106-02107/csvs/TSS12231-001_metric_summary.csv
Looking for TSS12232-001...
  Done -> /home/workspace/xenium/qc/EXP-02106-02107/csvs/TSS12232-001_metric_summary.csv
Looking for TSS10546-031...
  Done -> /home/workspace/xenium/qc/EXP-02106-02107/csvs/TSS10546-031_metric_summary.csv


In [ ]:
# ── Required ──────────────────────────────────────────────────────────────────
EXP_ID   = "EXP-02106-02107"
RUN_NAME = "051126"

# ── Optional (leave as None if not uploading to HISE) ─────────────────────────
uuid_list           = None   # e.g. ["967a55cb-...", "7660b4c5-..."]
project_short_name  = None   # e.g. "aifiDevTest"

# ── Derived (no need to edit) ─────────────────────────────────────────────────
BASE_DIR            = Path("/home/workspace/xenium/qc") / EXP_ID
all_qc_report_path  = BASE_DIR / "csvs"
output_dir          = BASE_DIR
title_description   = f"Xenium QC report for run name {EXP_ID}_{RUN_NAME}"
pdf_file_name       = f"Xenium_qc_report_{EXP_ID}.pdf"

In [8]:
import pandas as pd
import os
import datetime
import logging
import hisepy

### Option 1: Starting from metrics summary file UUIDs

In [4]:
def set_logging(log_dir, log_file_name):
    logger = logging.getLogger(__name__)
    fhandler = logging.FileHandler(filename=os.path.join(log_dir, log_file_name), mode='a')
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    fhandler.setFormatter(formatter)
    logger.addHandler(fhandler)
    logger.setLevel(logging.DEBUG)
    return logger

In [5]:
def download_files(logger, input_dir, uuid_list):
    #####################################################
    # This function downloads the files per uuids and   #
    # move the files to specified input folder          #
    #####################################################

    # Apply hisepy.reader.cache_files function to download the specified sample files #
    hisepy.reader.cache_files(uuid_list)

    # Move all the uuid folders from 'cache' folder to the 'input_dir'
    for each_uuid in uuid_list:
        logger.info("Copying file {0} from /home/jupyter/cache/ to {1}/......".format(each_uuid, input_dir))
        os.system('cp -r /home/jupyter/cache/{0} {1}/.'.format(each_uuid, input_dir))

In [ ]:
def create_aggregated_qc(logger, input_dir, output_dir, uuid_list):
    """Parse the per-sample QC report for each UUID and aggregate into one CSV."""
    run_name_list = []
    df_to_concat = []
    for each_uuid in uuid_list:
        per_qc_report_path = os.path.join(input_dir, each_uuid, 'metrics_summary.csv')

        if not os.path.exists(per_qc_report_path):
            logger.warning(f"Could not find QC report for uuid {each_uuid}, skipping it")
            continue
        if os.path.getsize(per_qc_report_path) == 0:
            logger.warning(f"QC report for uuid {each_uuid} is empty, skipping it")
            continue

        logger.info(f"Found QC report for uuid {each_uuid}")
        df_per_sample_qc = pd.read_csv(per_qc_report_path)
        df_to_concat.append(df_per_sample_qc)
        run_name_list.append(df_per_sample_qc.iloc[0]['run_name'])

    if not df_to_concat:
        raise RuntimeError("No QC reports were successfully loaded")

    df_aggregated_qc_report = pd.concat(df_to_concat, ignore_index=True)

    # If every sample shares a run_name, use it as the output prefix
    if len(set(run_name_list)) == 1:
        output_qc_file_name = f"{run_name_list[0]}_xenium_qc_report.csv"
    else:
        output_qc_file_name = 'xenium_qc_report.csv'

    df_aggregated_qc_report.to_csv(os.path.join(output_dir, output_qc_file_name), index=False)
    return output_qc_file_name

In [7]:
def upload_output_to_hise(project_short_name, uuid_list, title_description, output_qc_file_name, current_timestamp):
	hisepy.upload_files(files=[os.path.join(output_dir, output_qc_file_name)], 
	                    project=project_short_name, 
	                    title=title_description,
	                    destination = '{0}/xenium_qc_report'.format(current_timestamp),
	                    input_file_ids=uuid_list)


In [8]:
# Create a folder with timestamp in the current working directory #
current_timestamp = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
working_dir = os.path.join('/home/jupyter/execute_xenium_qc_module', current_timestamp)
if not os.path.exists(working_dir):
    os.makedirs(working_dir)

log_dir = os.path.join(working_dir, 'logs')
if not os.path.exists(log_dir):
    os.makedirs(log_dir)

input_dir = os.path.join(working_dir, 'input')
if not os.path.exists(input_dir):
    os.makedirs(input_dir)

output_dir = os.path.join(working_dir, 'output')
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

logger = set_logging(log_dir, 'xenium_qc_report.log')

# If uuid_list is not provided by user, print following warning #
if uuid_list and project_short_name and title_description:
    download_files(logger, input_dir, uuid_list)
    output_qc_file_name = create_aggregated_qc(logger, input_dir, output_dir, uuid_list)
    title_description_with_timestamp = '{0}-{1}'.format(title_description, current_timestamp) 
    upload_output_to_hise(project_short_name, uuid_list, title_description_with_timestamp, output_qc_file_name, current_timestamp)
else:
    if not uuid_list:
        print("Warning: please provide uuid_list")
    if not project_short_name:
        print("Warning: please provide project_short_name")
    if not title_description:
        print("Warning: please provide title_description")

In [9]:
os.listdir(all_qc_report_path)

['output-XETG00195__0033792__TIS08308-001__20240711__221925_metrics_summary.csv',
 '.ipynb_checkpoints',
 'output-XETG00123__0034044__TIS02784-001-001__20240711__221220_metrics_summary.csv',
 'output-XETG00123__0034044__TIS02795-001-009__20240711__221220_metrics_summary.csv',
 'output-XETG00195__0033752__TIS08305-001__20240711__221925_metrics_summary.csv',
 'output-XETG00123__0027461__TIS06055-001-011__20240606__220427_metrics_summary.csv',
 'output-XETG00195__0033752__TIS08306-001__20240711__221925_metrics_summary.csv',
 'output-XETG00123__0027459__TIS06055-001-009__20240606__220427_metrics_summary.csv',
 'output-XETG00123__0034036__TIS02805-001-009__20240711__221220_metrics_summary.csv',
 'output-XETG00195__0033792__TIS08307-001__20240711__221925_metrics_summary.csv',
 'output-XETG00123__0027459__TIS06055-001-010__20240606__220427_metrics_summary.csv']

### Option 2: Starting from metrics summary file's location instead of UUIDs 
#### (please skip if Option 1 (UUID method) has been run)

In [ ]:
csv_paths = sorted(Path(all_qc_report_path).glob('*.csv'))
if not csv_paths:
    raise FileNotFoundError(f"No CSVs found in {all_qc_report_path}")

df_aggregated_qc_report = pd.concat(
    [pd.read_csv(p) for p in csv_paths],
    ignore_index=True,
)
run_name_list = df_aggregated_qc_report['run_name'].tolist()

output_qc_file_name = 'xenium_qc_report.csv'
df_aggregated_qc_report.to_csv(Path(output_dir) / output_qc_file_name, index=False)

### Setting up data for visualizations

In [ ]:
plot_df = df_aggregated_qc_report.dropna(axis=1, how='all')
plot_df

In [11]:
run_name_list

['EXP-02106-02107_050726',
 'EXP-02106-02107_050726',
 'EXP-02106-02107_050726',
 'EXP-02106-02107_050726',
 'EXP-02106-02107_050726',
 'EXP-02106-02107_050726',
 'EXP-02106-02107_050726',
 'EXP-02106-02107_050726']

In [12]:
plot_df.columns

Index(['run_name', 'cassette_name', 'region_name', 'panel_name',
       'panel_design_id', 'predesigned_panel_id', 'region_area',
       'total_cell_area', 'total_high_quality_decoded_transcripts',
       'fraction_transcripts_decoded_q20',
       'fraction_predesigned_transcripts_decoded_q20',
       'fraction_custom_transcripts_decoded_q20',
       'nuclear_transcripts_per_100um2', 'decoded_transcripts_per_100um2',
       'adjusted_negative_control_probe_rate',
       'adjusted_negative_control_codeword_rate',
       'adjusted_genomic_control_probe_rate',
       'negative_control_probe_counts_per_control_per_cell',
       'genomic_control_probe_counts_per_control_per_cell',
       'estimated_number_of_false_positive_transcripts_per_cell',
       'estimated_number_of_false_positive_transcripts_per_cell_including_genomic_counts',
       'num_cells_detected', 'fraction_empty_cells', 'cells_per_100um2',
       'fraction_transcripts_assigned', 'median_genes_per_cell',
       'median_prede

In [13]:
import matplotlib.pyplot as plt
import plotly.express as px
from matplotlib.backends.backend_pdf import PdfPages

In [14]:
colors = [
'#9e0142','#d53e4f','#f46d43','#fdae61','#fee08b','#e6f598','#abdda4','#66c2a5','#3288bd','#5e4fa2'
]

In [15]:
col = ['#d53e4f','#66c2a5','#5e4fa2']

In [16]:
pdf_pages = PdfPages(pdf_file_name)

In [ ]:
def bar_plot(y_col, ylabel, title, color, df=None):
    df = plot_df if df is None else df
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(df['region_name'], df[y_col], color=color)
    plt.setp(ax.get_xticklabels(), rotation=60, ha='right')
    ax.set_xlabel('Tissue ID')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    fig.tight_layout()
    pdf_pages.savefig(fig)
    plt.close(fig)


def stacked_bar(value_cols, title, ylabel='Value', df=None, palette=None):
    df = plot_df if df is None else df
    palette = col if palette is None else palette
    fig, ax = plt.subplots(figsize=(9, 7))
    bottoms = pd.Series(0.0, index=df.index)
    for i, c in enumerate(value_cols):
        ax.bar(df['region_name'], df[c], bottom=bottoms, label=c, color=palette[i])
        bottoms = bottoms + df[c]
    ax.set_xlabel('Region Name')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(title='Metrics', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, linestyle='--', color='gray', alpha=0.6)
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    fig.tight_layout()
    pdf_pages.savefig(fig)
    plt.close(fig)

## Number of cells detected

In [ ]:
bar_plot('num_cells_detected', 'Number of cells', 'Number of cells detected by region', colors[6])

## Transcripts and Genes per Tissue

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(plot_df['region_name'], plot_df['median_transcripts_per_cell'], color=colors[7])
plt.plot(plot_df['region_name'], plot_df['median_genes_per_cell'], color=colors[1], marker='o', label='Median Genes per Cell')
plt.xticks(rotation=60, ha='right')
plt.xlabel('Tissue ID')
plt.ylabel('Median Transcripts per Cell')
plt.title('Median Transcripts per Cell by Region')
plt.legend()
plt.tight_layout()
pdf_pages.savefig()
plt.close()

## Estimated False Positive Transcripts

In [ ]:
bar_plot(
    'estimated_number_of_false_positive_transcripts_per_cell',
    'Number of False Positive Transcripts per Cell',
    'Estimated False Positive Transcripts per Region',
    colors[1],
)

## Decoded Transcripts

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(plot_df['region_name'], plot_df['fraction_transcripts_decoded_q20'], color=colors[8], label='Fraction Transcripts Decoded Q20', marker='^', s=120)
plt.scatter(plot_df['region_name'], plot_df['fraction_transcripts_assigned'], color=colors[2], label='Fraction Transcripts Assigned', marker='o', s=100)
plt.scatter(plot_df['region_name'], plot_df['fraction_empty_cells'], color=colors[7], label='Fraction Empty cells', marker='x', s=100)
plt.xticks(rotation=60, ha='right')
plt.xlabel('Region Name')
plt.ylabel('Fraction')
plt.title('Decoded Transcript Fractions')
plt.ylim(0, 1)
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.grid(True, linestyle='--', color='gray', alpha=0.6)
plt.tight_layout()
pdf_pages.savefig()
plt.close()

## Cells per 100um sq

In [ ]:
bar_plot('cells_per_100um2', 'N Cells', 'Cells per 100 um sq', colors[4])

## Segmentation Fractions and Counts

In [ ]:
stacked_bar(
    [
        'segmented_cell_boundary_frac',
        'segmented_cell_interior_frac',
        'segmented_cell_nuc_expansion_frac',
    ],
    title='Segmentation Fractions by Tissue',
)

In [ ]:
stacked_bar(
    [
        'segmented_cell_boundary_count',
        'segmented_cell_interior_count',
        'segmented_cell_nuc_expansion_count',
    ],
    title='Segmentation Counts by Tissue',
)

## Median Transcripts per panel

In [ ]:
plt.figure(figsize=(8, 5))
panels = plot_df['panel_name'].unique()

plt.boxplot(
    [plot_df.loc[plot_df['panel_name'] == p, 'median_transcripts_per_cell'] for p in panels],
    labels=panels,
    showfliers=False,
)

for i, panel in enumerate(panels, start=1):
    panel_data = plot_df.loc[plot_df['panel_name'] == panel, 'median_transcripts_per_cell']
    plt.scatter(
        [i] * len(panel_data),
        panel_data,
        color=colors[9],
        alpha=0.8,
        s=50,
        label='Points' if i == 1 else "_",
    )

plt.xlabel('Panel Name')
plt.ylabel('Median Transcripts per Cell')
plt.title('Median Transcripts per Cell by Panel')
plt.tight_layout()
pdf_pages.savefig()
plt.close()

In [25]:

plt.close()
pdf_pages.close()